In [ ]:
%load_ext autoreload
%autoreload 2

import requests
from datetime import datetime
import pandas as pd
from datetime import datetime, timedelta
import pylupnt as pnt
import numpy as np

In [ ]:
# Time Conversions test (from gnss_utils.py)
target_dt_utc = datetime(2025, 3, 1, 0, 00, 0)
print(f"Target UTC: {target_dt_utc}")
target_tai = pnt.datetime_to_tai(target_dt_utc, dt_timesys=pnt.UTC) - 18
print(f"Target TAI: {target_tai}")
converted_dt_utc = pnt.tai_to_datetime(target_tai, dt_timesys=pnt.UTC)
print(f"Converted UTC: {converted_dt_utc}")

week, day_of_week, sec_of_week, gps_dt = pnt.datetime_to_gpsweeks(
    target_dt_utc, timesys=pnt.UTC
)
print(
    f"From DateTime - GPS Week: {week}, Day of Week: {day_of_week}, Seconds of Week: {sec_of_week}, GPS Datetime: {gps_dt}"
)
week2, sec_into_week = pnt.tai_to_gps_weeks(target_tai)
print(f"From TAI - GPS Week: {week2}, Seconds into Week: {sec_into_week}")
tai2 = pnt.gps_weeks_to_tai(week2, sec_into_week)
print(f"Reconstructed TAI: {tai2}")
datetime2 = pnt.tai_to_datetime(tai2, dt_timesys=pnt.UTC)
print(f"Reconstructed UTC Datetime: {datetime2}")

tspan_tai = target_tai + np.linspace(0, 24 * 3600, 200)  # 24 hours span in TAI

## 1. Precise Ephemeris Loading

In [ ]:
import os

sp3 = pnt.SP3Loader(target_dt=target_dt_utc, sim_t=24 * 3600, dt_timesys=pnt.UTC)
print("SP3 satellites:", sp3.sats)
print("SP3 Epochs:", sp3.epochs)

In [ ]:
constellation = "E"  # 'G' for GPS, 'R' for GLONASS, 'E' for Galileo, etc.
prn = 10  # PRN number of the satellite

rv_prop, clock_bias = sp3.get_posvelclock(
    constellation, prn, target_tai, out_frame=pnt.ECEF, propagate=True
)
rv_interp, clock_bias_interp = sp3.get_posvelclock(
    constellation, prn, target_tai, out_frame=pnt.ECEF, propagate=False
)

week2, sec_into_week = pnt.tai_to_gps_weeks(target_tai)
print(f"From TAI - GPS Week: {week2}, Seconds into Week: {sec_into_week}")

print("RV Propagation for {0}{1:02d} at target TAI:".format(constellation, prn))
print("Position   [m]     : ", rv_prop[:3])
print("Velocity   [m/s]   : ", rv_prop[3:])
print("Clock Bias [micros] : ", clock_bias * 1e6)
print(" ")
print("RV Interpolation for {0}{1:02d} at target TAI:".format(constellation, prn))
print("Position   [m]     : ", rv_interp[:3])
print("Velocity   [m/s]   : ", rv_interp[3:])
print("Clock Bias [micros] : ", clock_bias_interp * 1e6)

plot entire constellation

In [ ]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# compute SP3 orbits and clock biases for all satellites
rv_sats_sp3, clock_sats_sp3, sats_list_sp3 = sp3.get_posvelclock_all(
    tspan_tai, out_frame=pnt.ECI, propagate=False
)

# plot orbits and clock biases
fig = go.Figure()

pnt.plot.plot_body(fig, pnt.EARTH, size_factor=5)

orbit_colors = {
    "G": "blue",  # GPS
    "E": "orange",  # GALILEO
    "C": "green",  # BEIDOU
    "R": "purple",  # GLONASS
    "J": "red",  # QZSS
}

gnss_consts = ["G", "E", "C", "R", "J"]  # GPS, GALILEO, BEIDOU, QZSS

# gnss
for gnss_const in gnss_consts:
    sat_idx = [i for i, sat in enumerate(sp3.sats) if sat.startswith(gnss_const)]
    pnt.plot.plot_orbits(
        fig, rv_sats_sp3[sat_idx, :, :], t=0, color=orbit_colors[gnss_const]
    )


pnt.plot.set_view(fig, -80, 20, 2.5)
fig.update_layout(showlegend=True, width=400, height=400)
fig.show()

# plot clock biases
Cms = 299792458  # Convert clock bias from seconds to meters
plt.figure(figsize=(6, 3))
for i, sat in enumerate(sats_list_sp3):
    plt.plot(
        tspan_tai,
        (clock_sats_sp3[i] - clock_sats_sp3[i][0]) * Cms,
        label=f"Satellite {sat}",
        alpha=0.7,
    )
plt.title("Clock Biases for Satellites")
plt.xlabel("Time (TAI)")
plt.ylabel("Clock Bias (m)")
plt.grid()
plt.show()

### 2. Broadcast Ephemeris

In [ ]:
brdc = pnt.BRDCLoader(target_dt=target_dt_utc, sim_t=2 * 3600, dt_timesys=pnt.UTC)

In [ ]:
# print the keys
brdc.nav_dict["E"][2].keys()

In [ ]:
constellation = "E"
prn = 12

rv_brdc, clock_bias_brdc = brdc.get_posvelclock(
    constellation, prn, target_tai, out_frame=pnt.ECEF
)

c = 299792458  # Speed of light in m/s

print(
    "RV Propagation for {0}{1:02d} at target TAI from BRDC:".format(constellation, prn)
)
print("Position   [m]     : ", rv_brdc[:3])
print("Velocity   [m/s]   : ", rv_brdc[3:])
print(f"Clock Bias [micros] : {clock_bias_brdc*c:.7f}")

# GALILEO PRN 10
# Rebecca: -207794.219558761
# Keidai:  -207793.8869066

# GALILEO PRN 12
# Rebecca: -144475.896
# Keidai: -144475.6294643

In [ ]:
# compute for all satellites
rv_sats_brdc, clock_sats_brdc, sats_list_brdc = brdc.get_posvelclock_all(
    tspan_tai, out_frame=pnt.ECI
)

### 3. Antenna Phase Offsets

In [ ]:
# compute SP3 orbits and clock biases for all satellites
tspan_tai = target_tai + np.linspace(0, 24 * 3600, 500)  # 24 hours span in TAI
rv_sats_sp3, clock_sats_sp3, sats_list_sp3 = sp3.get_posvelclock_all(
    tspan_tai, out_frame=pnt.ECEF, propagate=False
)
rv_sats_brdc, clock_sats_brdc, sats_list_brdc = brdc.get_posvelclock_all(
    tspan_tai, out_frame=pnt.ECEF
)

In [ ]:
from phase_center_offset import get_gnss_str, get_freq_str, ijk_to_ecef_rot
from tqdm import tqdm

atx = pnt.get_file_path("igs20.atx")
loader = pnt.ANTEXLoader(atx)
gnss_consts = ["GPS", "GALILEO", "QZSS"]
lent = len(tspan_tai)
print(f"Total time steps: {lent}")

pos_pco_dict = {}

for gnss_const in gnss_consts:
    gnss_str = get_gnss_str(gnss_const)
    freq_str = get_freq_str(gnss_const, signal_family=1)
    common_sats = set(sats_list_sp3) & set(sats_list_brdc)
    common_sats = [sat for sat in common_sats if sat.startswith(gnss_str)]
    nsat = len(common_sats)
    print(" Common satellites for {}: {}".format(gnss_const, common_sats))
    print(f"Processing {nsat} satellites")

    idx_sp3 = [sats_list_sp3.index(sat) for sat in common_sats]
    pos_sp3 = rv_sats_sp3[idx_sp3, :, :3]

    pco_neu = np.zeros((nsat, lent, 3))
    pco_ecef = np.zeros((nsat, lent, 3))

    for k, sat in enumerate(common_sats):
        prn = int(sat[1:])  # Extract PRN number from satellite name

        for t in tqdm(
            range(lent), desc=f"Processing {gnss_const} {prn} {freq_str} PCO"
        ):
            dt = pnt.tai_to_datetime(tspan_tai[t])
            try:
                pco_val = loader.get_pco(
                    dt, gnss_str, prn, freq=freq_str, freqcode=None
                )
            except Exception as e:
                if (
                    t == 0
                ):  # Only print the error for the first time step to avoid clutter
                    print(
                        f"Error retrieving PCO for {gnss_const} PRN {prn} at {dt}: {e}"
                    )
                pco_val = np.zeros(3)  # Default to zero if retrieval fails

            pco_neu[k, t, :] = pco_val
            rotmat = ijk_to_ecef_rot(
                tspan_tai[t], pos_sp3[k, t, :3]
            )  # Position vector from SP3)
            pco_ecef[k, t, :] = rotmat @ pco_neu[k, t, :]

    pos_pco_dict[gnss_const] = {
        "common_sats": common_sats,
        "pco_neu": pco_neu,
        "pco_ecef": pco_ecef,
    }

### 4. Plot difference between precise orbit and broadcast ephemeris

In [ ]:
# plot rv_sats_brdc - rv_sats_sp3
import matplotlib.pyplot as plt

# plot options
plot_beidou = False
use_rtn = True  # Set to True if you want to convert to RTN coordinates

if plot_beidou:
    consts = ["G", "E", "C", "J"]  # GPS, GALILEO, BEIDOU, QZSS
    consts_labels = ["GPS", "GALILEO", "BEIDOU", "QZSS"]
else:
    consts = ["G", "E", "J"]  # GPS, GALILEO, BEIDOU, QZSS
    consts_labels = ["GPS", "GALILEO", "QZSS"]


fig, axes = plt.subplots(len(consts), 4, figsize=(16, 3 * len(consts)))

tspan_hr = (tspan_tai - tspan_tai[0]) / 3600  # Convert TAI to hours

XYZ = ["X", "Y", "Z"]
Cm = 299792458.0  # Speed of light in meters per second

if use_rtn:
    XYZ = ["R", "T", "N"]  # RTN coordinates

for i, const in enumerate(consts):
    print(f"Processing constellation: {const}")
    common_sats = set(sats_list_sp3) & set(sats_list_brdc)
    common_sats = [sat for sat in common_sats if sat.startswith(const)]

    nsat = len(common_sats)
    lent = len(tspan_tai)

    idx_sp3 = [sats_list_sp3.index(sat) for sat in common_sats]
    idx_brdc = [sats_list_brdc.index(sat) for sat in common_sats]

    pos_brdc = rv_sats_brdc[idx_brdc, :, :3]
    vel_brdc = rv_sats_brdc[idx_brdc, :, 3:]
    clock_brdc = clock_sats_brdc[idx_brdc, :]

    pos_sp3 = rv_sats_sp3[idx_sp3, :, :3]
    vel_sp3 = rv_sats_sp3[idx_sp3, :, 3:]
    clock_sp3 = clock_sats_sp3[idx_sp3, :]

    # correct for PCO
    pco_ecef = pos_pco_dict[consts_labels[i]]["pco_ecef"]

    pos_diff = pos_brdc - (pos_sp3 + pco_ecef)  # N x T x 3 array
    clock_diff = clock_brdc - clock_sp3  # N x T array
    clock_diff_median = np.median(clock_diff.flatten())  # Convert to meters
    clock_diff -= clock_diff_median  # Remove median bias

    if use_rtn:
        for k in range(nsat):
            for t in range(lent):
                r1 = pos_sp3[k, t, :3]  # Position vector from SP3
                v1 = vel_sp3[k, t, :3]  # Velocity vector from SP3
                r2 = pos_brdc[k, t, :3]  # Position vector
                v2 = vel_brdc[k, t, :3]  # Velocity vector
                R = r1 / np.linalg.norm(r1)  # Radial unit vector
                N = np.cross(r1, v1)  # Normal unit vector
                N /= np.linalg.norm(N)  # Normalize
                T = np.cross(N, R)  # Tangential unit vector
                R_mat = np.vstack((R, T, N))
                pos_diff[k, t, :3] = (
                    R_mat @ pos_diff[k, t, :3]
                )  # Convert to RTN coordinates

    for k, sat in enumerate(common_sats):
        pos_diff_norm_m = np.linalg.norm(pos_diff[k], axis=1)  # Convert to meters
        clock_diff_m = clock_diff[k] * Cm  # Convert to meters
        if np.any(pos_diff_norm_m > 100.0):
            print(
                f"Warning: Large position differences: {np.max(pos_diff_norm_m)} m detected for {sat}"
            )
        if np.any(np.abs(clock_diff_m) > 20.0):
            print(
                f"Warning: Large clock differences: {np.min(clock_diff_m)} m detected for {sat}"
            )
        pos_diff_mean = np.mean(pos_diff_norm_m)
        clock_diff_mean = np.mean(np.abs(clock_diff_m))
        print(
            f"  {sat} - Mean Position Difference: {pos_diff_mean:.2f} m, Mean Clock Difference: {clock_diff_mean:.2f} m"
        )

    for j in range(3):
        ax = axes[i, j]
        ax.plot(tspan_hr, pos_diff[:, :, j].T)
        ax.set_title(f"{consts_labels[i]} Position Difference ({XYZ[j]})")
        ax.set_xlabel("Time (hr)")
        ax.set_ylabel("Position Difference (m)")
        ax.set_ylim(-5, 5)
        ax.set_xlim(0, tspan_hr[-1])
        ax.grid()

    ax = axes[i, 3]
    ax.plot(tspan_hr, clock_diff.T * Cm, label=f"{const} Clock Difference")
    ax.set_title(f"{consts_labels[i]} Clock Difference")
    ax.set_xlabel("Time (hr)")
    ax.set_ylabel("Clock Difference [m])")
    ax.set_ylim(-3, 3)
    ax.set_xlim(0, tspan_hr[-1])
    ax.grid()

fig.tight_layout()
plt.show()